# Regulus — end-to-end walkthrough

Describe an AI issue in plain language and Regulus returns the **regulatory
provisions that apply**, across frameworks, each with a **source citation** and
**cited cross-framework references**.

This notebook is a thin demo: every step is a one-line call into `regulus.demo`,
and each returns a small table. The real work lives in the `regulus` package
(`standards_loader`, `lookup`, `graph`, `graph_lookup`), so the notebook stays
readable and the logic stays testable and reusable.

**What it does, step by step:** set up → ingest real standards → look up applicable
provisions → build the regulatory knowledge graph → look up with cross-framework
crosswalks.

## 1. Setup

`ensure_gkn()` makes the [Geometric Knowledge Network](https://github.com/minw0607/geometric_knowledge_network)
importable — the installed package if present, otherwise the local sibling checkout —
so this notebook runs **without a `pip install`** in any kernel that has the usual
scientific stack (numpy, pandas, scikit-learn, networkx).

The default retriever is **TF-IDF** (no API keys). For higher-quality retrieval,
set `REGULUS_RETRIEVER=embedding` in a `.env` file (it reuses GKN's embedding
store — Azure/OpenAI or a local model); see `.env.example`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))   # make `regulus` importable

from regulus import demo
demo.ensure_gkn()
config = demo.config()
print('retriever:', config.retriever, '| top_k:', config.top_k)

## 2. Ingest the standards

Regulus fetches **real** regulatory text from official sources and splits it into
citable *provisions*:

- **EU AI Act** — the 113 articles, from EUR-Lex (© European Union; reuse with attribution).
- **NIST AI RMF 1.0** — the 72 subcategories, from the NIST PDF (U.S. Government work).

Sources are downloaded and **cached** under `data/standards_cache/` on first run
(then reused). Parsing the NIST PDF needs `pypdf`; if it isn't installed, Regulus
falls back to a **committed snapshot** of the NIST provisions, so this step works
either way. Each provision keeps its `source_url` — Regulus never returns an
uncited result.

In [ ]:
provisions = demo.load_provisions(config)
demo.provisions_summary(provisions)

## 3. Look up the applicable provisions

Given a free-text issue, Regulus retrieves the provisions whose text is the closest
match. `score` is a relative similarity (higher = closer); with TF-IDF the absolute
values are small — read them as a ranking, not a probability.

This is the **direct hit** — the provision the issue most looks like. The
cross-framework links come in Step 5.

**Try it:** edit the issue string below to your own observation.

In [ ]:
lookup = demo.baseline_lookup(provisions, config)
demo.lookup_table(lookup, 'We run real-time facial recognition in public spaces to assist law enforcement.')

## 4. Build the regulatory knowledge graph

The provisions become a graph:

- `Framework` **contains** `Provision` nodes;
- `Provision` **addresses** `RiskCategory` nodes (the seven NIST trustworthiness characteristics) — these tags are *keyword-derived and low-confidence*, meant for navigation;
- `Provision` ↔ `Provision` **crosswalk** edges link equivalent concerns across frameworks.

**Governance rule:** crosswalk edges come **only** from the curated, cited table
[`data/crosswalks/crosswalks.csv`](../data/crosswalks/crosswalks.csv) — never inferred by a model. To extend the graph, edit that
CSV (add rows with a `source`/citation) or add frameworks; nothing else changes.

In [ ]:
gl = demo.crosswalk_lookup(provisions, config)
demo.graph_stats(gl)

## 5. Look up with cross-framework crosswalks

The payoff: each applicable provision now carries the **risks** it addresses and
its **cited cross-framework references** — the same concern, linked to another
framework, with the mapping's citation. This is what a plain vector search cannot
do.

Read a row as: *"for this issue, this provision applies; it concerns these risks;
and here is the equivalent guidance in another framework (with its source)."*
Cross-framework references appear only when both frameworks are loaded and a
curated crosswalk connects them.

In [ ]:
demo.crosswalk_table(gl, 'Our credit model was deployed without testing for demographic bias.')

## 6. What's next

- **Evidence paths** — use GKN's multi-hop retriever + path explainer to return the full trail (issue → provision → crosswalk → provision).
- **Interpretation** — an LLM layer that turns the retrieved provisions + paths into a structured, cited answer (risks · standards · cross-refs · guidance).
- **Interface + evaluation** — a "submit an issue" UI and a benchmark of issue → expected-standards and crosswalk accuracy.

**Extending Regulus:** add crosswalk rows or frameworks in `data/`, or swap the seed
crosswalks for authoritative mappings. For embedding-quality retrieval, set
`REGULUS_RETRIEVER=embedding` in `.env`.